# PathSnap: 01. Exploratory Data Analysis (EDA)
### Distributed Map-Matching & Route Reconstruction

This notebook conducts comprehensive exploratory analysis of the raw trajectory and road network data:
1. **Dataset inventory & file schema verification**
2. **Missing value & data quality analysis**
3. **Trajectory length & sampling interval distributions**
4. **Spatial extent & coordinate bounds**
5. **Road network graph properties (nodes, arcs, connectivity)**
6. **Folium interactive visualization of GPS tracks against the road network**


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add root directory to sys.path
sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.data.preprocessing import haversine_distance, compute_bearing
from src.visualization.maps import create_folium_map

loader = TripLoader("../data")
print("Train trips:", len(loader.list_trips("train")))
print("Student-test trips:", len(loader.list_trips("student_test")))


## 1. Inspect Sample Trip Data
Each trip contains:
- `.arcs`: Connections between road nodes (from_node, to_node)
- `.nodes`: Road intersection coordinates (latitude, longitude)
- `.track`: GPS observations (latitude, longitude, timestamp)
- `.route`: Ground-truth traversed arc sequence


In [ ]:
sample_trip_id = loader.list_trips("student_test")[0]
trip = loader.load_trip(sample_trip_id, split="student_test")

print(f"Sample Trip ID: {trip['trip_id']}")
print(f"Arcs count: {len(trip['arcs'])}")
print(f"Nodes count: {len(trip['nodes'])}")
print(f"GPS Track points: {len(trip['track'])}")
print(f"Ground-truth Route segments: {len(trip['route']) if trip['route'] is not None else 0}")


## 2. GPS Sampling Intervals & Kinematic Properties


In [ ]:
track = trip["track"]
dts = np.diff(track[:, 2])
distances = [
    haversine_distance(track[i, 0], track[i, 1], track[i+1, 0], track[i+1, 1])
    for i in range(len(track) - 1)
]
speeds = np.array(distances) / np.maximum(1e-3, dts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dts, bins=30, color="#3B82F6", edgecolor="black")
axes[0].set_title("Sampling Interval (Δt) Distribution [s]")
axes[0].set_xlabel("Seconds")
axes[0].set_ylabel("Frequency")

axes[1].hist(speeds, bins=30, color="#10B981", edgecolor="black")
axes[1].set_title("Instantaneous Speed Distribution [m/s]")
axes[1].set_xlabel("Speed (m/s)")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()


## 3. Road Network Graph Topology


In [ ]:
arcs = trip["arcs"]
from_nodes = arcs[:, 0]
to_nodes = arcs[:, 1]

unique_nodes = np.unique(np.concatenate([from_nodes, to_nodes]))
print(f"Unique active road network intersection nodes: {len(unique_nodes)}")
print(f"Total directional arcs: {len(arcs)}")


## 4. Interactive Folium Trajectory Visualization


In [ ]:
# Create Folium visualization showing raw GPS points and ground truth route
ref_coords = []
if trip["route"] is not None:
    for arc_id in trip["route"]:
        if arc_id < len(arcs):
            u, v = arcs[arc_id, 0], arcs[arc_id, 1]
            ref_coords.append([float(trip["nodes"][u, 0]), float(trip["nodes"][u, 1])])
            ref_coords.append([float(trip["nodes"][v, 0]), float(trip["nodes"][v, 1])])

fmap = create_folium_map(
    track=trip["track"],
    ground_truth_coords=ref_coords,
    trip_id=sample_trip_id,
)
fmap.save("../results/plots/eda_sample_map.html")
print("Saved interactive EDA map to results/plots/eda_sample_map.html")
